# 05 Traditional Logit Versus Machine Learning Models

## Program Overview

This teaching program compares a traditional linear Logit model with two
machine learning classifiers: a Decision Tree and a Random Forest. The goal is
not to show that one family of methods is always better. Instead, students
learn how to compare behavioral interpretability, prediction accuracy,
probability quality, and feature importance using the same mode-choice data.

## Connection to Part 3 PPT

This notebook extends the Part 3 discussion after random utility theory,
Multinomial/Binary Logit, and maximum likelihood estimation. It also connects
compares discrete choice models with machine
learning models.

## Overall Task AI Prompt (English)

Use this notebook to compare a traditional Logit model and machine learning
models for transport mode choice. For each code cell, explain what the model is
trying to predict, which variables are used, how the result should be evaluated,
and what trade-off appears between behavioral explanation and predictive
performance.

## Learning Objectives

- Build a synthetic mode-choice data set with transport-relevant variables.
- Estimate a traditional linear Logit model as an interpretable benchmark.
- Train Decision Tree and Random Forest classifiers as machine learning models.
- Compare models using accuracy, log-loss, Brier score, and ROC-AUC.
- Interpret coefficient signs and feature importance without overclaiming causality.

## AI Prompt for This Code Cell

Generate a synthetic binary mode-choice data set. Identify the target variable and explain why each explanatory variable is relevant for travel behavior.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 600

time_diff = rng.normal(0, 12, n)
cost_diff = rng.normal(1.8, 3.0, n)
reliability_diff = rng.normal(0, 8, n)
transfers = rng.integers(0, 4, n)
income_high = rng.binomial(1, 0.42, n)

utility = (
    0.20
    - 0.075 * time_diff
    - 0.42 * cost_diff
    - 0.05 * reliability_diff
    - 0.30 * transfers
    + 0.38 * income_high
    + 0.018 * np.maximum(-time_diff, 0) * income_high
    - 0.18 * (transfers >= 2)
    + 0.10 * np.sin(cost_diff)
)

prob_car = 1.0 / (1.0 + np.exp(-utility))
choice_car = rng.binomial(1, prob_car)

data = pd.DataFrame(
    {
        "time_diff": time_diff,
        "cost_diff": cost_diff,
        "reliability_diff": reliability_diff,
        "transfers": transfers,
        "income_high": income_high,
        "choice_car": choice_car,
    }
)

summary = pd.Series(
    {
        "observations": len(data),
        "car_choices": int(data["choice_car"].sum()),
        "transit_choices": int((1 - data["choice_car"]).sum()),
        "car_choice_rate": data["choice_car"].mean(),
    }
)

print(summary)
data.head()

observations       600.000000
car_choices        199.000000
transit_choices    401.000000
car_choice_rate      0.331667
dtype: float64


,time_diff,cost_diff,reliability_diff,transfers,income_high,choice_car
0,-9.517470,4.555749,20.099126,1,0,0
1,2.886855,1.428869,2.707611,2,0,0
2,-22.755916,3.964754,4.987290,2,0,0
3,16.749261,-3.164017,-1.326764,1,0,1
4,7.659537,1.011615,12.634304,0,1,0


## Output Interpretation

The generated data set has 600 observations, with 199 car choices and 401 transit choices, so the car choice rate is about 33.17%. This class imbalance is moderate: transit is more common, but there are still enough car choices for model training. The variables are behaviorally meaningful. `time_diff`, `cost_diff`, and `reliability_diff` measure car disadvantage relative to the competing mode; `transfers` represents service inconvenience; and `income_high` captures a socio-economic difference. The simulated utility includes mostly linear transport effects plus a small interaction and nonlinear term, so it is suitable for comparing an interpretable Logit model with more flexible machine learning models.

## AI Prompt for This Code Cell

Split the data into training and test sets, then estimate a traditional linear Logit model. Interpret the coefficient signs as behavioral sensitivities.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

train, test = train_test_split(
    data,
    test_size=0.30,
    random_state=3,
    stratify=data["choice_car"],
)

traditional_features = [
    "time_diff",
    "cost_diff",
    "reliability_diff",
    "transfers",
]

traditional_logit = LogisticRegression(max_iter=1000, solver="lbfgs")
traditional_logit.fit(train[traditional_features], train["choice_car"])

logit_coefficients = pd.Series(
    traditional_logit.coef_[0],
    index=traditional_features,
    name="coefficient",
)

print(f"Training observations: {len(train)}")
print(f"Test observations: {len(test)}")
print(f"Intercept: {traditional_logit.intercept_[0]:.4f}")
print(logit_coefficients.round(4))

Training observations: 420
Test observations: 180
Intercept: 0.3514
time_diff          -0.0874
cost_diff          -0.4252
reliability_diff   -0.0829
transfers          -0.4742
Name: coefficient, dtype: float64


## Output Interpretation

The split creates 420 training observations and 180 test observations. The traditional Logit model estimates an intercept of about 0.3514 and four negative coefficients: `time_diff = -0.0874`, `cost_diff = -0.4252`, `reliability_diff = -0.0829`, and `transfers = -0.4742`. These signs are exactly what a transport choice interpretation would expect. If car becomes slower, more expensive, less reliable, or requires more transfers relative to the competing mode, the utility of choosing car decreases. This is the main strength of the traditional model: each coefficient can be connected directly to a behavioral mechanism in the random utility framework.

## AI Prompt for This Code Cell

Train two machine learning classifiers on the same prediction task. Use the additional socio-economic feature and compare their role to the traditional Logit specification.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

ml_features = traditional_features + ["income_high"]

decision_tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=20,
    random_state=3,
)
random_forest = RandomForestClassifier(
    n_estimators=120,
    max_depth=5,
    min_samples_leaf=12,
    random_state=3,
)

decision_tree.fit(train[ml_features], train["choice_car"])
random_forest.fit(train[ml_features], train["choice_car"])

print("Machine learning models trained:")
print(f"- Decision Tree features: {ml_features}")
print(f"- Random Forest features: {ml_features}")

Machine learning models trained:
- Decision Tree features: ['time_diff', 'cost_diff', 'reliability_diff', 'transfers', 'income_high']
- Random Forest features: ['time_diff', 'cost_diff', 'reliability_diff', 'transfers', 'income_high']


## Output Interpretation

Both machine learning models use five predictors: the four transport variables plus `income_high`. Unlike the traditional Logit model, the Decision Tree and Random Forest do not estimate one linear utility coefficient for each variable. They learn split rules and combinations of variables that improve prediction. This flexibility can be useful when the behavioral relationship is nonlinear or includes interactions. However, the cost is that interpretation is less direct: a tree split or forest importance score does not have the same behavioral meaning as a Logit coefficient in a utility equation.

## AI Prompt for This Code Cell

Evaluate all three models on the test set. Compare accuracy, log-loss, Brier score, and ROC-AUC rather than relying on only one metric.

In [4]:
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

models = {
    "Traditional linear Logit": (traditional_logit, traditional_features),
    "Decision Tree": (decision_tree, ml_features),
    "Random Forest": (random_forest, ml_features),
}

rows = []
for model_name, (model, features) in models.items():
    probability = model.predict_proba(test[features])[:, 1]
    prediction = model.predict(test[features])
    rows.append(
        {
            "model": model_name,
            "features_used": len(features),
            "accuracy": accuracy_score(test["choice_car"], prediction),
            "log_loss": log_loss(test["choice_car"], probability),
            "brier_score": brier_score_loss(test["choice_car"], probability),
            "roc_auc": roc_auc_score(test["choice_car"], probability),
        }
    )

comparison = pd.DataFrame(rows)
comparison.round(4)

,model,features_used,accuracy,log_loss,brier_score,roc_auc
0,Traditional linear Logit,4,0.8000,0.4014,0.1320,0.8833
1,Decision Tree,5,0.7722,0.5483,0.1774,0.7761
2,Random Forest,5,0.7889,0.4511,0.1450,0.8576


## Output Interpretation

The test-set comparison shows that the traditional linear Logit performs best in this simulated example: accuracy = 0.8000, log-loss = 0.4014, Brier score = 0.1320, and ROC-AUC = 0.8833. The Random Forest is second, with accuracy = 0.7889, log-loss = 0.4511, Brier score = 0.1450, and ROC-AUC = 0.8576. The Decision Tree performs worst, with accuracy = 0.7722 and log-loss = 0.5483. Lower log-loss and Brier score are better, while higher accuracy and ROC-AUC are better. The result is a useful teaching point: machine learning does not automatically outperform a traditional model. When the underlying choice process is close to a linear utility model, the Logit model can be both interpretable and highly predictive.

## AI Prompt for This Code Cell

Inspect feature importance for the machine learning models and compare it with the Logit coefficient interpretation.

In [5]:
tree_importance = pd.Series(
    decision_tree.feature_importances_,
    index=ml_features,
    name="decision_tree_importance",
).sort_values(ascending=False)

forest_importance = pd.Series(
    random_forest.feature_importances_,
    index=ml_features,
    name="random_forest_importance",
).sort_values(ascending=False)

importance_table = pd.concat([tree_importance, forest_importance], axis=1).fillna(0)
importance_table.round(4)

,decision_tree_importance,random_forest_importance
cost_diff,0.4299,0.3894
time_diff,0.2545,0.3475
reliability_diff,0.2539,0.1593
transfers,0.0617,0.0942
income_high,0.0000,0.0097


## Output Interpretation

The Decision Tree gives the largest importance to `cost_diff` (0.4299), followed by `time_diff` (0.2545) and `reliability_diff` (0.2539). The Random Forest also ranks `cost_diff` highest (0.3894), followed by `time_diff` (0.3475), `reliability_diff` (0.1593), `transfers` (0.0942), and `income_high` (0.0097). This broadly agrees with the traditional Logit result: time and cost are central predictors. But feature importance is not the same as a marginal utility coefficient. It tells us which variables helped prediction inside the tree-based models, not the direction or exact behavioral sensitivity of the choice probability.

## AI Prompt for This Code Cell

Write a short modeling conclusion that separates prediction quality from behavioral interpretation.

In [6]:
best_by_accuracy = comparison.sort_values("accuracy", ascending=False).iloc[0]
best_by_log_loss = comparison.sort_values("log_loss", ascending=True).iloc[0]

print(
    f"Best model by accuracy: {best_by_accuracy['model']} "
    f"({best_by_accuracy['accuracy']:.2%})"
)
print(
    f"Best model by log-loss: {best_by_log_loss['model']} "
    f"({best_by_log_loss['log_loss']:.4f})"
)

conclusion = (
    "In this teaching example, the traditional linear Logit model is the best "
    "test-set performer and also has the clearest behavioral interpretation. "
    "The machine learning models are still useful benchmarks because they test "
    "whether additional flexibility improves prediction."
)

print(conclusion)

Best model by accuracy: Traditional linear Logit (80.00%)
Best model by log-loss: Traditional linear Logit (0.4014)
In this teaching example, the traditional linear Logit model is the best test-set performer and also has the clearest behavioral interpretation. The machine learning models are still useful benchmarks because they test whether additional flexibility improves prediction.


## Output Interpretation

The final output identifies the Traditional linear Logit as the best model by both accuracy (80.00%) and log-loss (0.4014). This conclusion is intentionally nuanced. The lesson is not that traditional models always beat machine learning; rather, the appropriate model depends on the data-generating process, the evaluation metric, and the purpose of the analysis. If the teaching goal is behavioral explanation and policy interpretation, the Logit model is attractive because coefficient signs and magnitudes map to utility theory. If the goal is pure prediction in a complex nonlinear setting, machine learning models may become stronger, but they must prove that advantage on held-out data.